In [2]:
#Nhiệm vụ 5
import pandas as pd
try:
    df_log = pd.read_csv('clean_attendance_log.csv')
    df_student = pd.read_csv('clean_student_class.csv')
    df_schedule = pd.read_csv('clean_class_schedule.csv')
    
    master_df = pd.merge(df_log, df_student, on='student_id', how='left')
    master_df = pd.merge(master_df, df_schedule, on='class_id', how='left')
    print("Đã tải dữ liệu thành công!")
except FileNotFoundError:
    print("Lỗi: Cần file csv sạch từ Task 1.")

print("\n SỐ BUỔI VẮNG THEO SV x LỚP ")
pivot_student_class = master_df[master_df['status'] == 'Absent'].pivot_table(
    index=['student_id', 'full_name'],
    columns='class_id',
    values='date',
    aggfunc='count',
    fill_value=0
)
print(pivot_student_class.head())


print("\n XU HƯỚNG VẮNG THEO MÔN x THỨ ")
pivot_course_weekday = master_df[master_df['status'] == 'Absent'].pivot_table(
    index='course_name',
    columns='weekday',
    values='student_id',
    aggfunc='count',
    fill_value=0
)

# Sắp xếp lại thứ tự cột cho đúng chuẩn
week_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
existing_days = [day for day in week_order if day in pivot_course_weekday.columns]
pivot_course_weekday = pivot_course_weekday[existing_days]
print(pivot_course_weekday)


print("\n STACK & UNSTACK ")
# Chuyển sang dạng dài (Stack)
stacked_data = pivot_course_weekday.stack().reset_index()
stacked_data.columns = ['Môn học', 'Thứ', 'Số lượng vắng']
print("\n> Dữ liệu sau khi Stack:")
print(stacked_data.head())

# Chuyển lại dạng rộng (Unstack)
unstacked_data = stacked_data.set_index(['Môn học', 'Thứ']).unstack()
print("\n> Dữ liệu sau khi Unstack:")
print(unstacked_data.head())


print("\n KẾT QUẢ PHÂN TÍCH")
# Tìm môn vắng nhiều nhất
total_absent_by_course = pivot_course_weekday.sum(axis=1).sort_values(ascending=False)
top_course = total_absent_by_course.index[0]
top_course_count = total_absent_by_course.iloc[0]

# Tìm ngày vắng nhiều nhất
total_absent_by_day = pivot_course_weekday.sum(axis=0).sort_values(ascending=False)
top_day = total_absent_by_day.index[0]
top_day_count = total_absent_by_day.iloc[0]

print(f"- Môn có lượng vắng cao nhất: {top_course} ({top_course_count} lượt).")
print(f"- Ngày dễ nghỉ học nhất: {top_day} ({top_day_count} lượt).")

Đã tải dữ liệu thành công!

 SỐ BUỔI VẮNG THEO SV x LỚP 
class_id                 HP101_01
student_id full_name             
sv_003     Le Quang Huy         1

 XU HƯỚNG VẮNG THEO MÔN x THỨ 
weekday           Monday
course_name             
Lap Trinh Python       1

 STACK & UNSTACK 

> Dữ liệu sau khi Stack:
            Môn học     Thứ  Số lượng vắng
0  Lap Trinh Python  Monday              1

> Dữ liệu sau khi Unstack:
                 Số lượng vắng
Thứ                     Monday
Môn học                       
Lap Trinh Python             1

 KẾT QUẢ PHÂN TÍCH
- Môn có lượng vắng cao nhất: Lap Trinh Python (1 lượt).
- Ngày dễ nghỉ học nhất: Monday (1 lượt).
